# Baseline 1: Pure LLM Conversational QA
## GraphRAG Benchmark — Bloomberg Financial News

**Project**: GraphRAG vs Vanilla RAG vs Pure LLM  
**Role**: This notebook answers financial questions using only the LLM's parametric memory — no retrieval, no context, no knowledge graph.

---

**What is being compared:**

| System | Context source | This notebook? |
|--------|---------------|----------------|
| GraphRAG | Subgraph retrieved from knowledge graph | No — your pipeline |
| Vanilla RAG | Top-k chunks from vector index | Baseline 2 |
| Pure LLM | None — LLM memory only | **This notebook** |

**The comparison question:** When you give an LLM structured graph context (GraphRAG), does it answer financial questions better than when it gets raw chunk context (Vanilla RAG) or no context at all (Pure LLM)?

All three systems answer the **same 25 questions** from the **same Bloomberg dataset**. Answers are scored on the same metrics so the numbers are directly comparable.

## 1. Environment Setup

**Why Mistral NeMo 12B Instruct:**  
Same model used across all three baselines — the LLM is held constant so that the only variable is the context source.  
Any performance difference between the three methods is attributable to retrieval quality, not to the model itself.

In [1]:
import subprocess, sys
packages = [
    "datasets","transformers","torch","bitsandbytes",
    "accelerate","pandas","numpy","tqdm","rouge-score",
]
for pkg in packages:
    subprocess.check_call([sys.executable,"-m","pip","install",pkg,"-q"])
print("Setup complete.")

KeyboardInterrupt: 

## 2. Imports and Configuration

In [ ]:
import json, os, re, time
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, pipeline
)
from rouge_score import rouge_scorer

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

CFG = {
    "model_name"       : "mistralai/Mistral-Nemo-Instruct-2407",
    "dataset_name"     : "XJCEO/bloomberg_financial_news_120k",
    "n_articles"       : 5000,    # same subset as all other notebooks
    "max_new_tokens"   : 400,
    "temperature"      : 0.1,
    "qa_set_path"      : "/content/qa_eval_set.json",
    "output_path"      : "/content/purell_qa_results.json",
}
print("\nConfiguration loaded.")

## 3. Load Bloomberg Dataset

We load the same 5000-article subset used across all notebooks.  
For the Pure LLM baseline, we do not feed any article text to the model — we load the dataset only to generate the shared QA evaluation set (Section 4) and to verify questions are answerable from it.

In [ ]:
print("Loading Bloomberg Financial News dataset...")
dataset = load_dataset(CFG["dataset_name"], split="train")
df = dataset.to_pandas()
TEXT_COL = "Article" if "Article" in df.columns else df.select_dtypes("object").columns[0]
df = df[df[TEXT_COL].apply(lambda x: isinstance(x,str) and len(x.strip())>30)].reset_index(drop=True)
df["article_id"] = df.index
df = df.head(CFG["n_articles"]).copy()
print(f"Subset loaded: {len(df)} articles")
print(df[["article_id","Headline"]].head(5).to_string())

## 4. Shared QA Evaluation Set

**Why auto-generate QA pairs from the dataset:**  
We have no hand-annotated financial QA benchmark for Bloomberg data.  
Instead we sample 25 articles, extract the key fact from each article using the LLM, and turn it into a question.  
The article text becomes the reference answer source.

**Why this is fair across all three baselines:**  
The questions are derived from articles that exist in the dataset.  
- Vanilla RAG has access to the full article corpus and can retrieve the right article.  
- Pure LLM has no access — it must rely on parametric memory alone.  
- GraphRAG has the knowledge graph built from these same articles.

This design maximally stresses the difference between the three approaches.

**This cell generates the QA set once and saves it to `/content/qa_eval_set.json`.**  
All three baseline notebooks load this same file — run this cell only in Baseline 1.

In [ ]:
# ── Sample 25 diverse articles for QA generation
# Use a fixed stride across the 5000 articles for diversity
qa_article_ids = list(range(0, 5000, 200))[:25]
qa_articles = df[df["article_id"].isin(qa_article_ids)].reset_index(drop=True)

print(f"Articles selected for QA generation:")
for _, row in qa_articles.iterrows():
    print(f"  [{row['article_id']:4d}] {row['Headline'][:70]}")

In [ ]:
# ── Load model early to generate QA set
print(f"Loading {CFG['model_name']} with 4-bit quantization...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.bfloat16,
    bnb_4bit_use_double_quant = True,
)
tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])
model = AutoModelForCausalLM.from_pretrained(
    CFG["model_name"],
    quantization_config = bnb_config,
    device_map          = "auto",
    trust_remote_code   = True,
)
model.eval()

pipe = pipeline(
    "text-generation", model=model, tokenizer=tokenizer,
    max_new_tokens=CFG["max_new_tokens"], temperature=CFG["temperature"],
    do_sample=False,
)
vram = torch.cuda.memory_allocated()/1e9 if torch.cuda.is_available() else 0
print(f"Model loaded. VRAM: {vram:.2f} GB")

In [ ]:
QA_GEN_SYSTEM = """You are a financial QA dataset creator.
Given a financial news article, generate ONE specific factual question that can only be answered by reading this article,
and provide the ideal 2-3 sentence reference answer.

Output ONLY valid JSON in this exact format:
{"question": "...", "reference_answer": "...", "topic": "earnings|ma|personnel|market|policy|other"}"""

def generate_qa_pair(article_text: str, headline: str) -> dict | None:
    """Generate a question + reference answer from one article."""
    prompt = f"""Headline: {headline}

Article:
{article_text[:1000]}

Generate a specific factual question and reference answer. Output ONLY the JSON:"""

    messages = [
        {"role":"system","content":QA_GEN_SYSTEM},
        {"role":"user","content":prompt},
    ]
    try:
        raw = pipe(messages)[0]["generated_text"]
        reply = raw[-1].get("content","") if isinstance(raw,list) else str(raw)
        reply = re.sub(r"```json\s*","",reply)
        reply = re.sub(r"```\s*","",reply).strip()
        match = re.search(r"\{.*\}", reply, re.DOTALL)
        if match:
            parsed = json.loads(match.group())
            if "question" in parsed and "reference_answer" in parsed:
                return parsed
    except Exception:
        pass
    return None

# Generate QA pairs
print(f"Generating {len(qa_articles)} QA pairs from sampled articles...")
qa_eval_set = []

for _, row in tqdm(qa_articles.iterrows(), total=len(qa_articles), desc="QA generation"):
    pair = generate_qa_pair(str(row[TEXT_COL]), str(row["Headline"]))
    if pair:
        qa_eval_set.append({
            "qa_id"           : len(qa_eval_set),
            "article_id"      : int(row["article_id"]),
            "headline"        : str(row["Headline"]),
            "question"        : pair["question"],
            "reference_answer": pair["reference_answer"],
            "topic"           : pair.get("topic","other"),
        })
        print(f"  [{len(qa_eval_set):2d}] Q: {pair['question'][:70]}")

print(f"\nGenerated {len(qa_eval_set)} QA pairs.")

# Save for all three baselines to share
with open(CFG["qa_set_path"],"w") as f:
    json.dump(qa_eval_set, f, indent=2)
print(f"QA set saved: {CFG['qa_set_path']}")

## 5. Pure LLM Inference

For each question we send **only the question** to the LLM — no article text, no retrieved chunks, no graph context.  
The model must rely entirely on what it learned during pre-training.

**Why this is a meaningful baseline:**  
Mistral NeMo 12B was trained on large amounts of financial text from the internet.  
It may answer some financial questions correctly from memory.  
But it cannot answer questions about specific events in the Bloomberg dataset that were not in its training data.  
This gap — between what the LLM knows and what the dataset contains — is exactly what GraphRAG and Vanilla RAG are designed to fill.

In [ ]:
INFERENCE_SYSTEM = """You are a knowledgeable financial analyst.
Answer the financial question clearly and concisely in 2-4 sentences.
Base your answer on your knowledge of financial markets, companies, and economics.""""

def ask_pure_llm(question: str) -> str:
    """Ask the LLM a question with no context — pure parametric memory."""
    messages = [
        {"role":"system","content":INFERENCE_SYSTEM},
        {"role":"user","content":question},
    ]
    try:
        raw   = pipe(messages)[0]["generated_text"]
        reply = raw[-1].get("content","") if isinstance(raw,list) else str(raw)
        return reply.strip()
    except Exception as e:
        return f"[ERROR: {e}]"

# Load QA set (in case re-running this cell independently)
with open(CFG["qa_set_path"]) as f:
    qa_eval_set = json.load(f)

print(f"Running Pure LLM inference on {len(qa_eval_set)} questions...")
print()

results = []
t_start = time.time()

for qa in tqdm(qa_eval_set, desc="Pure LLM QA"):
    answer = ask_pure_llm(qa["question"])
    results.append({
        "qa_id"           : qa["qa_id"],
        "article_id"      : qa["article_id"],
        "topic"           : qa["topic"],
        "question"        : qa["question"],
        "reference_answer": qa["reference_answer"],
        "predicted_answer": answer,
        "method"          : "pure_llm",
        "context_used"    : None,
    })

elapsed = time.time() - t_start
print(f"\nInference done: {len(results)} answers in {elapsed:.0f}s ({elapsed/len(results):.1f}s each)")

# Preview 3 answers
print()
for r in results[:3]:
    print(f"Q: {r['question']}")
    print(f"A: {r['predicted_answer'][:200]}")
    print(f"Ref: {r['reference_answer'][:150]}")
    print()

## 6. Evaluation

**Two complementary metrics:**

**ROUGE-L** (automated):  
Measures longest common subsequence overlap between the predicted and reference answer.  
Fast and reproducible — good for catching completely wrong or completely right answers.  
Limitation: a factually correct answer phrased differently scores poorly. We use it for relative comparison, not absolute quality.

**LLM-as-Judge** (qualitative):  
We ask the same Mistral model to score each answer on three dimensions (1-5 each):
- **Relevance** — does the answer address the question?
- **Factual grounding** — are the claims plausible and accurate?
- **Completeness** — is the answer sufficiently detailed?

This captures answer quality that ROUGE-L misses, and is the standard evaluation approach when no gold labels are available.

**Why these metrics work for a three-way comparison:**  
All three systems (Pure LLM, Vanilla RAG, GraphRAG) are scored with the same rubric on the same questions.  
The absolute scores matter less than the relative ranking.

In [ ]:
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

JUDGE_SYSTEM = """You are an objective evaluator of financial QA answers.
Score the predicted answer on three criteria (1-5 each):
- relevance: does it answer the question asked?
- factual_grounding: are the claims accurate and supported?
- completeness: is the answer sufficiently detailed?

Output ONLY valid JSON: {"relevance": 3, "factual_grounding": 4, "completeness": 2, "reasoning": "brief explanation"}"""

def llm_judge(question:str, reference:str, predicted:str) -> dict:
    """Score a predicted answer using the LLM as judge."""
    prompt = f"""Question: {question}

Reference answer: {reference}

Predicted answer: {predicted}

Score the predicted answer. Output ONLY the JSON:"""
    messages = [
        {"role":"system","content":JUDGE_SYSTEM},
        {"role":"user","content":prompt},
    ]
    try:
        raw   = pipe(messages)[0]["generated_text"]
        reply = raw[-1].get("content","") if isinstance(raw,list) else str(raw)
        reply = re.sub(r"```json\s*","",reply)
        reply = re.sub(r"```\s*","",reply).strip()
        match = re.search(r"\{.*\}",reply,re.DOTALL)
        if match:
            scores = json.loads(match.group())
            return {
                "relevance"         : float(scores.get("relevance",0)),
                "factual_grounding" : float(scores.get("factual_grounding",0)),
                "completeness"      : float(scores.get("completeness",0)),
                "reasoning"         : scores.get("reasoning",""),
            }
    except Exception:
        pass
    return {"relevance":0,"factual_grounding":0,"completeness":0,"reasoning":"parse error"}

# ── Score all answers
print(f"Evaluating {len(results)} answers (ROUGE-L + LLM judge)...")
print()

for r in tqdm(results, desc="Evaluating"):
    # ROUGE-L
    rouge = scorer.score(r["reference_answer"], r["predicted_answer"])
    r["rouge_l"] = round(rouge["rougeL"].fmeasure, 4)

    # LLM judge
    judge_scores = llm_judge(r["question"], r["reference_answer"], r["predicted_answer"])
    r["judge"]   = judge_scores
    r["judge_avg"] = round(
        (judge_scores["relevance"] + judge_scores["factual_grounding"] + judge_scores["completeness"]) / 3, 4
    )

# ── Print results
print()
print("=== PURE LLM QA EVALUATION RESULTS ===")
print(f"{'#':>3}  {'Topic':10s}  {'ROUGE-L':>8}  {'Judge Avg':>9}  {'Question (truncated)'}")
print("-"*75)
for r in results:
    print(f"{r['qa_id']:3d}  {r['topic']:10s}  {r['rouge_l']:8.4f}  {r['judge_avg']:9.4f}  {r['question'][:42]}")

avg_rouge = np.mean([r["rouge_l"]   for r in results])
avg_judge = np.mean([r["judge_avg"] for r in results])
avg_rel   = np.mean([r["judge"]["relevance"]          for r in results])
avg_fact  = np.mean([r["judge"]["factual_grounding"]  for r in results])
avg_comp  = np.mean([r["judge"]["completeness"]       for r in results])

print()
print("=== AGGREGATE METRICS ===")
print(f"  ROUGE-L (avg)           : {avg_rouge:.4f}")
print(f"  Judge score (avg)       : {avg_judge:.4f} / 5.0")
print(f"    Relevance             : {avg_rel:.4f}")
print(f"    Factual grounding     : {avg_fact:.4f}")
print(f"    Completeness          : {avg_comp:.4f}")

# By topic
from collections import defaultdict
topic_scores = defaultdict(list)
for r in results:
    topic_scores[r["topic"]].append(r["judge_avg"])
print("\n  By topic:")
for topic, scores in sorted(topic_scores.items()):
    print(f"    {topic:12s}: {np.mean(scores):.4f}")

## 7. Save Results

The output file contains all answers and scores in a schema shared across all three baselines.  
The final comparison notebook loads this alongside Vanilla RAG and GraphRAG results to build the comparison table.

In [ ]:
summary = {
    "method"          : "Pure LLM (Mistral NeMo 12B — no context)",
    "n_questions"     : len(results),
    "avg_rouge_l"     : float(avg_rouge),
    "avg_judge_score" : float(avg_judge),
    "avg_relevance"   : float(avg_rel),
    "avg_factual"     : float(avg_fact),
    "avg_completeness": float(avg_comp),
    "per_question"    : results,
}

with open(CFG["output_path"],"w") as f:
    json.dump(summary, f, indent=2)

print(f"Results saved: {CFG['output_path']}")
print(f"Size: {os.path.getsize(CFG['output_path'])/1024:.1f} KB")

In [ ]:
# ── Save QA set and results to Google Drive
from google.colab import drive
drive.mount('/content/drive')
import shutil

for fpath in [CFG["qa_set_path"], CFG["output_path"]]:
    if os.path.exists(fpath):
        dest = f"/content/drive/MyDrive/{os.path.basename(fpath)}"
        shutil.copy(fpath, dest)
        print(f"Saved to Drive: {os.path.basename(fpath)}")

## Summary

| Aspect | Choice | Reason |
|--------|--------|--------|
| Model | Mistral NeMo 12B Instruct (4-bit NF4) | Same across all 3 baselines — isolates context source as the variable |
| Context | None | Baseline measures what the LLM knows without any retrieval |
| QA set | 25 auto-generated pairs from Bloomberg articles | No gold labels available; LLM-generated questions grounded in the dataset |
| ROUGE-L | Automated overlap metric | Fast, reproducible, good for relative ranking |
| LLM-as-Judge | 3-dimension scoring (relevance / factual / completeness) | Captures quality ROUGE-L misses; standard when no gold labels exist |

**Expected result vs GraphRAG:**  
Pure LLM will likely score well on general financial questions (market trends, definitions, well-known companies) but poorly on event-specific questions about articles in the Bloomberg dataset that postdate its training or were not widely covered — because it has no access to the dataset at all.

**Output files:**
- `/content/qa_eval_set.json` — shared QA set used by all three baselines
- `/content/purell_qa_results.json` — answers + scores for this baseline